In [ ]:
# --- Colab bootstrap: in Colab, clone + install the repo; locally, do nothing ---
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/CKmellow/per3-genotype-clock-gene-timing.git"
BRANCH = "main"  # change to your branch name to test it in Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo = Path("/content/per3-genotype-clock-gene-timing")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    src = str(repo / "src")
else:  # local: find the repo root (works from notebooks/ or notebooks/stages/)
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), here)
    src = str(root / "src")
if src not in sys.path:
    sys.path.insert(0, src)  # makes `import per3clock` work without a kernel restart

from per3clock import config  # noqa: E402

config.set_seeds()
config.ensure_dirs()
print("per3clock root:", config.ROOT, "| Colab:", IN_COLAB)

# Stage 4 · Exploratory data analysis

*Owner: Part 2 (see issue #2).*

## Minimum evidence (lecturer's brief)
*Brief, stage 4: "Describe and visualize the data; look for patterns, outliers." Minimum evidence: "at least three informative plots, interpretation."*

- [ ] **At least three informative plots** (more is fine; uninformative plots don't count)
- [ ] **An interpretation for each plot**: what pattern or outlier it shows and why it matters for the question
- [ ] Look explicitly for patterns (time of day, subject, genotype, condition) and outliers
- [ ] Short interpretation of your own results
- [ ] Report what the data does **and does not** show. A null, negative or inconclusive result is a valid result: report it with its uncertainty.

## Contract
**Inputs:** `data.analysis_samples()` (shared default; use `for_phase_analysis=True` for plots against `circadian_phase_h`), `data.load_clock_genes()` (one probe per gene), expression matrix via `data.collapse_probes_to_genes(data.load_expression_matrix())`.

**Implement in `per3clock.eda`:** `plot_clock_gene_profiles`, `pca_samples`, `plot_pca`, `plot_sample_correlation`.

**Outputs:** figures `eda_clock_gene_profiles.png`, `eda_pca_subject.png`, `eda_pca_timepoint.png`, `eda_sample_correlation.png` (via `plotting.save_fig`, 300 dpi); also `eda_clock_gene_profiles_secondary.png`, `eda_pca_overview.png`, `eda_pca_phase.png`, `eda_pca_condition.png`, and tables `eda_clock_gene_profiles.csv`, `eda_subject_timing.csv`, `eda_pca_variance.csv`, `eda_pca_factor_r2.csv`.

**EDA is descriptive.** Genotype groups are shown as mean ± SE, with no hypothesis tests; those are Part 3's job.

In [ ]:
import pandas as pd
from IPython.display import display

from per3clock import config, data, eda, plotting, qc

plotting.set_style()
pd.set_option("display.width", 140)
FIG_DPI = 300  # report figures

meta = data.load_metadata()
main = data.analysis_samples(metadata=meta)                              # 24 subjects, 221 samples
phase_set = data.analysis_samples(metadata=meta, for_phase_analysis=True)  # 23 subjects (no AF0182)
clock = data.load_clock_genes()                                          # one probe per gene
clock_main = clock[clock.sample_id.isin(main.sample_id)]
print(len(main), "main samples;", len(phase_set), "samples for phase analyses")

## 4.1 Clock-gene profiles by genotype

Each panel shows one gene: thin lines are individual subjects, thick lines the genotype mean ± SE across subjects (subjects are the unit; a bin is drawn only if it has ≥ 3 subjects). Expression is **centred within each subject** (subject mean subtracted per gene), which removes between-person level differences so the curves show timing; raw levels are in Stage 3 (`qc_biology_checks`).

- **Top row, clock time.** Sampling started between 14:00 and 18:15 depending on the participant and ran across midnight, so clock time is *unwrapped* (`eda.unwrap_clock_time`) and binned in 3-h windows; grouping by T1–T10 would mix clock times up to 4 h apart.
- **Bottom row, circadian phase** (`circadian_phase_h`, already in 3-h bins). Phase 0 is *probably* the plasma-melatonin midpoint; that is inferred from the data (implied mean 04:10 vs the published 04:15), not confirmed. AF0182 is dropped from this row because of its inconsistent phase labels.

In [ ]:
fig = eda.plot_clock_gene_profiles(clock_main, genes=config.GENES_PRIMARY,
                                   time_col=("clock_time_h", "circadian_phase_h"), center=True)
plotting.save_fig(fig, "eda_clock_gene_profiles", dpi=FIG_DPI)

fig = eda.plot_clock_gene_profiles(clock_main, genes=("CLOCK", "CRY1"),
                                   time_col=("clock_time_h", "circadian_phase_h"), center=True)
plotting.save_fig(fig, "eda_clock_gene_profiles_secondary", dpi=FIG_DPI)

profiles = pd.concat([
    eda.genotype_profile_table(clock_main, config.GENES_PRIMARY + ("CLOCK", "CRY1"), tc, center=True)
    .assign(time_axis=tc) for tc in ("clock_time_h", "circadian_phase_h")])
profiles.round(4).to_csv(config.TABLES_DIR / "eda_clock_gene_profiles.csv", index=False)

# Schedules differ between people: implied wake time and phase-0 (melatonin proxy) clock time
timing = eda.subject_timing_summary(main)
timing.round(3).to_csv(config.TABLES_DIR / "eda_subject_timing.csv")
display(timing.groupby("per3_genotype")[["wake_clock_h", "phase0_clock_h"]]
        .agg(["count", "mean", "std", "min", "max"]).round(2).T)
print("Descriptive only (decimal clock hours; AF0182 has no phase-0 value). No test here: see Part 3.")

**Figure `eda_clock_gene_profiles`.**

- **Rhythms.** PER2 and NR1D1 show clear daily rhythms in nearly every subject. NR1D1 rises through the evening and is highest around 01:00–05:00 clock time (phase −3 to 0 h); PER2 is highest a few hours later (about 04:00–08:00, phase 0 to +3 h); both reach a trough in the afternoon. The same shape appears against circadian phase, so the curves follow the internal clock and are not just a drift across 40 h awake. PER3 has a smaller amplitude (about ±0.2 log2) and noisier individual series, highest around 03:00–08:00 (phase 0 to +3 h). One 4/4 subject has a single large spike at its last sample, which is visible in the plot and should be watched in Part 3.
- **Genotype (descriptive only, no test).** The 4/4 and 5/5 mean curves largely overlap, and their ±SE bands meet or overlap in most bins. On *clock time*, the 5/5 mean reaches its highest bin one bin (3 h) earlier than 4/4 for PER2 and NR1D1, in the hypothesised direction, and in the same bin for PER3. On *circadian phase* that difference disappears (NR1D1, PER3) or reverses (PER2). The highest bin is a crude summary at 3-h resolution, so these are impressions, not estimates.
- **Secondary genes (`eda_clock_gene_profiles_secondary`).** CLOCK is flat on both axes, and CRY1 shows at most a weak, broad rise. This supports keeping them secondary.

**Schedules and melatonin timing differ by genotype (table above, descriptive).** On average the 5/5 participants woke about 1.2 h earlier than the 4/4 participants, and their implied phase-0 clock time (the melatonin proxy) was also about 1.2 h earlier. That fits the pattern in the curves: an earlier 5/5 peak in clock time that disappears when time is measured from each person's own melatonin rhythm. Whether this holds up, and how large it is with a CI, is objectives 2–3 and Part 3's job. With 11 vs 13 subjects only shifts larger than roughly 1–2 h are likely to be detectable.

## 4.2 PCA

PCA of the 221 main samples on the expressed genes from Stage 3 (`qc.expressed_genes`: one probe per gene, background filter), each gene centred, not scaled. How much of each PC is explained by subject, genotype, timepoint and circadian phase is measured with η² (share of the PC's variance explained by the grouping; descriptive, no p-values). Subject is nested in genotype, so genotype's η² can never exceed subject's; with 24 subjects η²(subject) would be ≈ 0.1 by chance alone.

In [ ]:
expr = data.load_expression_matrix()
annot = data.load_probe_annotation()
expressed, threshold = qc.expressed_genes(expr, annot, main.sample_id)
print(f"{len(expressed):,} expressed genes (threshold log2 {threshold:.3f}) x {expressed.shape[1]} samples")

scores, explained = eda.pca_samples(expressed, n_components=10)
r2 = eda.pca_factor_r2(scores.iloc[:, :6], main, explained=explained.iloc[:6])
explained.to_frame().assign(cumulative=explained.cumsum()).round(4).to_csv(config.TABLES_DIR / "eda_pca_variance.csv")
r2.round(4).to_csv(config.TABLES_DIR / "eda_pca_factor_r2.csv")
display(explained.to_frame().assign(cumulative=explained.cumsum()).T.round(3))
display(r2.round(3))

plotting.save_fig(eda.plot_pca(scores, main, "subject", explained=explained), "eda_pca_subject", dpi=FIG_DPI)
plotting.save_fig(eda.plot_pca(scores, main, "timepoint", explained=explained), "eda_pca_timepoint", dpi=FIG_DPI)
fig = eda.plot_pca_grid(scores, main, ("per3_genotype", "subject", "timepoint", "clock_time_h"),
                        explained=explained, suptitle="PCA of the main set (Sleep Extension, 221 samples)")
plotting.save_fig(fig, "eda_pca_overview", dpi=FIG_DPI)
# circadian phase: AF0182 SE has inconsistent phase labels, so its samples are not shown
in_phase = scores.index.isin(phase_set.sample_id)
fig = eda.plot_pca(scores[in_phase], phase_set, "circadian_phase_h", explained=explained,
                   title="PCA coloured by circadian phase (23 subjects; AF0182 not shown)")
plotting.save_fig(fig, "eda_pca_phase", dpi=FIG_DPI)

**Side by side: Sleep Extension and Sleep Restriction.** Not part of the main analysis. One PCA on both default sample sets (SE 221 + SR 217 samples, same expressed genes) shows whether the sleep condition moves samples more than the person does.

In [ ]:
sr = data.analysis_samples("Sleep Restriction", metadata=meta)
both = pd.concat([main, sr], ignore_index=True)
genes_both = data.collapse_probes_to_genes(expr, annot).loc[expressed.index, both.sample_id]
scores_b, explained_b = eda.pca_samples(genes_both, n_components=5)
display(eda.pca_factor_r2(scores_b, both, ("subject", "sleep_condition", "timepoint"),
                          explained=explained_b).round(3))
fig = eda.plot_pca_grid(scores_b, both, ("sleep_condition", "subject"), explained=explained_b,
                        suptitle="Side by side, NOT the main analysis: Sleep Extension + Sleep Restriction (438 samples)")
plotting.save_fig(fig, "eda_pca_condition", dpi=FIG_DPI)

**PCA (`eda_pca_subject`, `eda_pca_timepoint`, `eda_pca_overview`, `eda_pca_phase`).** The first two PCs explain 21.7 % and 11.9 % of the variance, and the first ten 64 %. **Subject dominates.** Subject explains 82 % of PC1 and 91 % of PC2 (η²), and 77 % of the variance across PCs 1–6 (weighted). Samples from the same person cluster together; for example, the low PC2 cluster is one 4/4 subject (AF0010), not a group of bad arrays. **Time of day is a minor source of variation:** timepoint explains about 4 % and circadian phase about 4 % across PCs 1–6, with at most 7 % on any one PC (PC3). That is expected, because only a minority of genes are rhythmic in blood, and it is why Part 3 should model each subject's rhythm within that subject rather than compare raw levels between people. **Genotype** explains about 3 %, essentially what is carried along by subject (subject is nested in genotype), and no genotype separation is visible.

**Side by side with Sleep Restriction (`eda_pca_condition`).** On the 438 samples, subject explains 75 % of the leading PCs and sleep condition 0.3 %. A week of restricted sleep does not move the whole transcriptome as much as the difference between people does; the main and sensitivity sets are comparable in overall structure.

## 4.3 Sample correlation

Pearson correlation between all pairs of main-set samples on the expressed genes, ordered by genotype, subject and timepoint, with colour bars for genotype and subject.

In [ ]:
fig = eda.plot_sample_correlation(expressed, main)
plotting.save_fig(fig, "eda_sample_correlation", dpi=FIG_DPI)

display(eda.correlation_summary(expressed, main).round(4))

**Figure `eda_sample_correlation`.** The heatmap shows bright blocks on the diagonal, one per subject. Pairs of samples from the same person correlate more strongly (median r 0.988) than pairs from different people (0.967–0.968), confirming the PCA. Pairs from *different genotypes* correlate as well as pairs of different people within the same genotype (median 0.966 vs 0.968): there is no genome-wide genotype signature. A few subjects (e.g. AF0010) are slightly less similar to everyone else, which is the same person-level structure seen on PC2, not a sample problem.

## Interpretation

- **What dominates the variation?** The person. Subject explains about 77 % of the leading principal components; time of day (timepoint or circadian phase) explains about 4 %, genotype about 3 % (carried by subject), and sleep condition under 1 %. Any analysis of timing must therefore work *within subjects* (per-subject cosinor fits, as planned for Part 3), not compare raw expression across people.
- **Do the clock genes look rhythmic?** Yes for the primary genes. PER2 and NR1D1 clearly are, and PER3 is too but with a smaller amplitude and more noise. They peak in the early morning, in the expected order (NR1D1 before PER2/PER3), on both clock time and circadian phase. CLOCK and CRY1 show little or no rhythm.
- **Any visible genotype difference?** Nothing large. The genotype mean curves overlap. On clock time the 5/5 group looks up to one 3-h bin earlier for PER2 and NR1D1, matching the 5/5 group's earlier wake times and earlier melatonin timing (both about 1.2 h earlier on average); on circadian phase the curves line up. These are visual, descriptive impressions with no test and no CI.
- **What the EDA does not show.** It cannot say whether genotype shifts peak time: that needs per-subject peak estimates, a between-group comparison with a 95 % CI, and the melatonin-relative analysis (Part 3, objectives 1–3). A null or inconclusive answer there would be consistent with these plots, given ~12 subjects per genotype and 3-h sampling.